In [2]:
!pip install -r requirements.txt

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 29.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.4/795.4 kB 115.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 54.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 92.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 50.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 104.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 109.6 MB/s  0:00:00
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting uninstall: click0m╺━━━━━━━━━━━━━━━━━━━━━━━━━  

In [3]:
import os
# Block TensorFlow import — not needed, and it causes crashes
# due to NumPy 2.x / pyarrow ABI mismatch in this Anaconda env
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import gc, re
from transformers import pipeline

gc.collect()
print("Cell 1 OK ✓")

/home/psylab-6028/fsl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cell 1 OK ✓


In [4]:
# Load aspect classifier (zero-shot) — force CPU to avoid MPS crashes
print("Loading aspect model...")
aspect_clf = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device="cpu",
)
print("Aspect model loaded ✓")

Loading aspect model...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 12102.87it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Aspect model loaded ✓


In [5]:
# Load Hebrew sentiment model — force CPU
print("Loading sentiment model...")
sent_clf = pipeline(
    "text-classification",
    model="dicta-il/dictabert-sentiment",
    top_k=None,
    device="cpu",
)
print("Sentiment model loaded ✓")

Loading sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10338.15it/s]


Sentiment model loaded ✓


In [6]:
ASPECTS = ["כאב", "שינה", "אוכל"]

def split_sentences_he(text: str):
    parts = re.split(r"[.!?。\n]+", text)
    return [p.strip() for p in parts if p.strip()]

def aspect_scores(text: str):
    out = aspect_clf(text, candidate_labels=ASPECTS, multi_label=True)
    return dict(zip(out["labels"], out["scores"]))

def sentiment_probs(text: str):
    scores = sent_clf(text)[0]
    d = {x["label"].lower(): float(x["score"]) for x in scores}
    return {
        "pos": d.get("positive", 0.0),
        "neg": d.get("negative", 0.0),
        "neu": d.get("neutral", 0.0),
    }

def analyze(text: str, eps: float = 1e-9):
    sents = split_sentences_he(text)
    if not sents:
        sents = [text]

    per_sent = []
    for s in sents:
        a = aspect_scores(s)
        p = sentiment_probs(s)
        per_sent.append((a, p))

    rel = {asp: sum(a.get(asp, 0.0) for a, _ in per_sent) / len(per_sent) for asp in ASPECTS}

    overall = {
        k: sum(p[k] for _, p in per_sent) / len(per_sent)
        for k in ["pos", "neg", "neu"]
    }

    by_asp = {}
    for asp in ASPECTS:
        wsum = sum(a.get(asp, 0.0) for a, _ in per_sent)
        by_asp[asp] = {
            k: sum(a.get(asp, 0.0) * p[k] for a, p in per_sent) / (wsum + eps)
            for k in ["pos", "neg", "neu"]
        }

    string_aspects = ", \n\t".join(f"{asp}: {rel[asp]:.2f}" for asp in ASPECTS)
    
    string_by_asp = "\n\t".join(
        f"{asp}: {by_asp[asp]}" for asp in ASPECTS
    )
    message = f"Relevance:\n\t{string_aspects}\nOverall sentiment: {overall}\nSentiment by aspect:\n\t{string_by_asp}"

    return message

print("Functions defined ✓")

Functions defined ✓


In [7]:
# Test
print(analyze("ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב מעולה."))

Relevance:
	כאב: 0.50, 
	שינה: 0.38, 
	אוכל: 0.52
Overall sentiment: {'pos': 0.4999914933538889, 'neg': 0.5000056027174651, 'neu': 2.9104123768775025e-06}
Sentiment by aspect:
	כאב: {'pos': 0.002811803087834231, 'neg': 0.9971845383186636, 'neu': 3.677902086818545e-06}
	שינה: {'pos': 0.001266643726061135, 'neg': 0.998729695035166, 'neu': 3.680287327777866e-06}
	אוכל: {'pos': 0.965781594493546, 'neg': 0.03421620669228059, 'neu': 2.1913783533593e-06}


In [8]:
print(analyze("ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב ממוצעת."))

Relevance:
	כאב: 0.81, 
	שינה: 0.39, 
	אוכל: 0.52
Overall sentiment: {'pos': 2.1764481630270893e-06, 'neg': 0.49999976236426846, 'neu': 0.4999980860272899}
Sentiment by aspect:
	כאב: {'pos': 2.0923581255420203e-06, 'neg': 0.6185445062649283, 'neu': 0.38145342454011083}
	שינה: {'pos': 1.8261449683913219e-06, 'neg': 0.993834753102124, 'neu': 0.006163439897872061}
	אוכל: {'pos': 2.506860696450909e-06, 'neg': 0.03420534307340242, 'neu': 0.9657921780864865}


---
## 🔬 Heavy Model Comparison (250 GB RAM / 32 cores)

Models under test:

| Task | Lightweight (current) | Heavy |
|---|---|---|
| Zero-shot aspects | `mDeBERTa-v3-base-mnli-xnli` (~280 M) | `xlm-roberta-large-xnli` (~560 M) |
| Sentiment | `dictabert-sentiment` (~130 M) | `avichr/heBERT_sentiment_analysis` (~110 M) |
| **Bonus** | — | `Qwen/Qwen2.5-3B-Instruct` LLM-prompted (~3 B) |

In [10]:
import torch, time

# Leverage all 32 cores
torch.set_num_threads(32)
torch.set_num_interop_threads(4)
print(f"PyTorch threads: intra={torch.get_num_threads()}, inter={torch.get_num_interop_threads()}")
print(f"Torch version: {torch.__version__}")
gc.collect()
print("Threading configured ✓")

PyTorch threads: intra=32, inter=4
Torch version: 2.9.1+cu128
Threading configured ✓


In [11]:
# ── Heavy zero-shot model: XLM-RoBERTa Large (~560M params) ──
print("Loading XLM-RoBERTa Large zero-shot model...")
t0 = time.time()
aspect_clf_heavy = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device="cpu",
)
print(f"XLM-RoBERTa Large loaded in {time.time()-t0:.1f}s ✓")

Loading XLM-RoBERTa Large zero-shot model...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 23244.19it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-RoBERTa Large loaded in 24.5s ✓


In [12]:
# ── Alternative Hebrew sentiment: HeBERT ──
print("Loading HeBERT sentiment model...")
t0 = time.time()
sent_clf_hebert = pipeline(
    "sentiment-analysis",
    model="avichr/heBERT_sentiment_analysis",
    top_k=None,
    device="cpu",
)
print(f"HeBERT sentiment loaded in {time.time()-t0:.1f}s ✓")

Loading HeBERT sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 44864.83it/s]
BertForSequenceClassification LOAD REPORT from: avichr/heBERT_sentiment_analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HeBERT sentiment loaded in 9.4s ✓


In [17]:
!pip install accelerate

In [ ]:
# ── Bonus: Qwen 2.5 3B Instruct — LLM-prompted aspect + sentiment ──
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading Qwen2.5-3B-Instruct (this may take a minute)...")
t0 = time.time()
qwen_model_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tok = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float32,
)
qwen_model = qwen_model.to("cpu").eval()
print(f"Qwen 3B loaded in {time.time()-t0:.1f}s  ✓  "
      f"(~{sum(p.numel() for p in qwen_model.parameters())/1e9:.1f}B params)")

Loading Qwen2.5-3B-Instruct (this may take a minute)...


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [ ]:
# ── Analysis functions using HEAVY models ──

def aspect_scores_heavy(text: str):
    """XLM-RoBERTa Large zero-shot."""
    out = aspect_clf_heavy(text, candidate_labels=ASPECTS, multi_label=True)
    return dict(zip(out["labels"], out["scores"]))

def sentiment_probs_hebert(text: str):
    """HeBERT sentiment (labels: positive / negative / neutral)."""
    scores = sent_clf_hebert(text)[0]
    d = {x["label"].lower(): float(x["score"]) for x in scores}
    return {
        "pos": d.get("positive", d.get("pos", 0.0)),
        "neg": d.get("negative", d.get("neg", 0.0)),
        "neu": d.get("neutral", d.get("neu", 0.0)),
    }

def analyze_heavy(text: str, eps: float = 1e-9):
    """Same pipeline as analyze() but with heavier models."""
    sents = split_sentences_he(text)
    if not sents:
        sents = [text]

    per_sent = []
    for s in sents:
        a = aspect_scores_heavy(s)
        p = sentiment_probs_hebert(s)
        per_sent.append((a, p))

    rel = {asp: sum(a.get(asp, 0.0) for a, _ in per_sent) / len(per_sent) for asp in ASPECTS}

    overall = {
        k: sum(p[k] for _, p in per_sent) / len(per_sent)
        for k in ["pos", "neg", "neu"]
    }

    by_asp = {}
    for asp in ASPECTS:
        wsum = sum(a.get(asp, 0.0) for a, _ in per_sent)
        by_asp[asp] = {
            k: round(sum(a.get(asp, 0.0) * p[k] for a, p in per_sent) / (wsum + eps), 4)
            for k in ["pos", "neg", "neu"]
        }

    string_aspects = ", \n\t".join(f"{asp}: {rel[asp]:.2f}" for asp in ASPECTS)
    string_by_asp = "\n\t".join(f"{asp}: {by_asp[asp]}" for asp in ASPECTS)
    return f"Relevance:\n\t{string_aspects}\nOverall sentiment: {overall}\nSentiment by aspect:\n\t{string_by_asp}"


# ── Qwen LLM-prompted analysis ──

import json as _json

def analyze_qwen(text: str):
    """Use Qwen 3B to extract aspects + sentiment via a structured prompt."""
    system = (
        "You are a Hebrew NLP assistant. "
        "Given a Hebrew text, identify which of these aspects are discussed: כאב, שינה, אוכל. "
        "For each relevant aspect, classify the sentiment as positive, negative, or neutral. "
        "Reply ONLY with a valid JSON object like: "
        '{"aspects": [{"aspect": "שינה", "sentiment": "negative", "confidence": 0.9}, ...],'
        ' "overall_sentiment": "negative"}'
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text},
    ]
    prompt = qwen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    answer = qwen_tok.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    # Try to parse JSON, fall back to raw text
    try:
        parsed = _json.loads(answer)
        return _json.dumps(parsed, ensure_ascii=False, indent=2)
    except _json.JSONDecodeError:
        return answer

print("Heavy analysis functions defined ✓")

In [ ]:
# ── Side-by-side comparison ──
test_texts = [
    "ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב מעולה.",
    "ישנתי רע, אתמול בלילה התעוררתי מלא פעמים והמיטה הכאיבה לי בגב.  אכלתי ארוחת ערב ממוצעת.",
    "הכאבים שלי השתפרו מאוד היום, אני מרגיש הרבה יותר טוב. ישנתי מצוין.",
    "האוכל היה נורא, הבטן כאבה לי כל הלילה ולא הצלחתי לישון.",
]

for i, text in enumerate(test_texts, 1):
    print(f"{'='*80}")
    print(f"TEXT {i}: {text[:80]}...")
    print()

    t0 = time.time()
    res_light = analyze(text)
    dt_light = time.time() - t0

    t0 = time.time()
    res_heavy = analyze_heavy(text)
    dt_heavy = time.time() - t0

    print(f"── Lightweight ({dt_light:.2f}s) ──")
    print(res_light)
    print()
    print(f"── Heavy ({dt_heavy:.2f}s) ──")
    print(res_heavy)
    print()

print("Comparison done ✓")

In [ ]:
# ── Qwen 3B LLM comparison ──
for i, text in enumerate(test_texts, 1):
    print(f"{'='*80}")
    print(f"TEXT {i}: {text[:80]}...")
    print()
    t0 = time.time()
    res_qwen = analyze_qwen(text)
    dt = time.time() - t0
    print(f"── Qwen 3B ({dt:.1f}s) ──")
    print(res_qwen)
    print()

print("Qwen comparison done ✓")

In [ ]:
# ── Cleanup: free heavy models if needed ──
# Uncomment to free ~5 GB of RAM
# del aspect_clf_heavy, sent_clf_hebert, qwen_model, qwen_tok
# gc.collect()
# print("Heavy models freed ✓")